# Feature Engineering - Module 1B (sujet Khady KAMA)

À exécuter sur Cifer-echantillon-strat.csv, après EDA et contrôle qualité.
 
7 points du sujet :
  1. Ratio de solde émetteur : (oldbalanceOrg - newbalanceOrig) / (amount + 1)
  2. Différence de solde destinataire : newbalanceDest - oldbalanceDest vs amount
  3. Flag d'anomalie de montant : montant = 100% du solde émetteur
  4. Features temporelles : heure du jour, jour de la semaine, période jour/nuit
  5. Encodage one-hot du type de transaction
  6. Features d'agrégation par agent (si le volume le permet)
  7. Suppression des identifiants nameOrig/nameDest
 
Auteur : Rasmané

## 0. CHARGEMENT DE L'ÉCHANTILLON

In [ ]:
CHEMIN_ENTREE = r"C:\Users\hp\Documents\Fraude_detection\data\Cifer-echantillon-strat.csv"
CHEMIN_SORTIE = r"C:\Users\hp\Documents\Fraude_detection\data\Cifer-echantillon-features.csv"
 
DTYPES = {
    "step": "int16",
    "type": "category",
    "amount": "float32",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8",
    "isFlaggedFraud": "int8",
}
 
data1 = pd.read_csv(CHEMIN_ENTREE, dtype=DTYPES)
print(f"Échantillon chargé : {data1.shape[0]:,} lignes, {data1.shape[1]} colonnes")
print(f"Colonnes initiales : {data1.columns.tolist()}\n")

## POINT 1 — RATIO DE SOLDE ÉMETTEUR

In [ ]:
# (oldbalanceOrg - newbalanceOrig) / (amount + 1)
# Interprétation : si la transaction est cohérente, ce ratio doit être
# proche de 1 (tout le montant débité correspond à la baisse de solde).
# Un ratio très différent de 1 (0, négatif, ou très grand) signale une
# incohérence potentielle -> pattern intéressant pour la fraude.
print("=" * 70)
print("POINT 1 : Ratio de solde émetteur")
print("=" * 70)
 
data1["ratio_solde_orig"] = (
    (data1["oldbalanceOrg"] - data1["newbalanceOrig"]) / (data1["amount"] + 1)
)
 
print(data1["ratio_solde_orig"].describe())
print("\nComparaison fraude / non-fraude :")
print(data1.groupby("isFraud")["ratio_solde_orig"].describe())

## POINT 2 — DIFFÉRENCE DE SOLDE DESTINATAIRE vs AMOUNT

In [ ]:
# newbalanceDest - oldbalanceDest devrait être ~= amount si la réception
# est cohérente. On calcule à la fois la différence brute et l'écart
# par rapport à amount (l'écart est la vraie feature d'anomalie).
print("\n" + "=" * 70)
print("POINT 2 : Différence de solde destinataire vs amount")
print("=" * 70)
 
data1["diff_solde_dest"] = data1["newbalanceDest"] - data1["oldbalanceDest"]
data1["ecart_solde_dest_vs_amount"] = data1["diff_solde_dest"] - data1["amount"]
 
print(data1[["diff_solde_dest", "ecart_solde_dest_vs_amount"]].describe())
print("\nComparaison fraude / non-fraude (écart vs amount) :")
print(data1.groupby("isFraud")["ecart_solde_dest_vs_amount"].describe())

## POINT 3 — FLAG D'ANOMALIE DE MONTANT (100% du solde émetteur)

In [ ]:
# Transaction où le montant représente (quasi) 100% du solde disponible
# avant transaction -> pattern classique de "vidage de compte".
# Tolérance de 1% pour absorber les arrondis flottants.
print("\n" + "=" * 70)
print("POINT 3 : Flag d'anomalie de montant (vidage de compte)")
print("=" * 70)
 
TOLERANCE_PCT = 0.01  # 1% de tolérance
data1["flag_montant_egal_solde"] = (
    (data1["oldbalanceOrg"] > 0) &
    (data1["amount"] >= data1["oldbalanceOrg"] * (1 - TOLERANCE_PCT)) &
    (data1["amount"] <= data1["oldbalanceOrg"] * (1 + TOLERANCE_PCT))
).astype("int8")
 
nb_flag = data1["flag_montant_egal_solde"].sum()
print(f"Transactions flaggées (montant ~= solde émetteur) : {nb_flag:,} "
      f"({nb_flag/len(data1)*100:.2f}%)")
taux_fraude_flag = data1.loc[data1["flag_montant_egal_solde"]==1, "isFraud"].mean()
taux_fraude_sans_flag = data1.loc[data1["flag_montant_egal_solde"]==0, "isFraud"].mean()
print(f"Taux de fraude avec ce flag : {taux_fraude_flag*100:.2f}%")
print(f"Taux de fraude sans ce flag : {taux_fraude_sans_flag*100:.2f}%")
if taux_fraude_sans_flag > 0:
    print(f"Ratio : {taux_fraude_flag/taux_fraude_sans_flag:.1f}x plus de fraude avec ce flag")

## POINT 4 — FEATURES TEMPORELLES

In [ ]:
# step = 1 heure, simulation sur 30 jours (744 steps)
# heure_du_jour : 0-23 (step % 24)
# jour_semaine : 0-6 (jour_simulation % 7) -- Lundi=0 arbitraire, car pas
#                de date calendaire réelle, seulement un jour "relatif"
# periode_jour_nuit : nuit = 22h-6h (créneau à risque plus élevé en général)
print("\n" + "=" * 70)
print("POINT 4 : Features temporelles")
print("=" * 70)
 
data1["heure_du_jour"] = (data1["step"] % 24).astype("int8")
data1["jour_simulation"] = (data1["step"] // 24).astype("int16")
data1["jour_semaine"] = (data1["jour_simulation"] % 7).astype("int8")
data1["est_nuit"] = (
    (data1["heure_du_jour"] >= 22) | (data1["heure_du_jour"] < 6)
).astype("int8")
 
print("Aperçu des features temporelles créées :")
print(data1[["step", "heure_du_jour", "jour_simulation", "jour_semaine", "est_nuit"]].head())
 
print("\nTaux de fraude par période jour/nuit :")
print(data1.groupby("est_nuit")["isFraud"].agg(["mean", "count"]))
 
print("\nTaux de fraude par jour de la semaine (0=jour relatif 0, etc.) :")
print(data1.groupby("jour_semaine")["isFraud"].mean())
 
print("\nNote méthodologique : le dataset ne fournit pas de date calendaire")
print("réelle, seulement 'step' (heures écoulées depuis le début de la")
print("simulation). 'jour_semaine' est donc un jour RELATIF au début de la")
print("simulation, pas un vrai jour de semaine (lundi/mardi/...). À préciser")
print("dans le rapport pour éviter toute confusion sur l'interprétation.")

## POINT 5 — ENCODAGE ONE-HOT DU TYPE DE TRANSACTION

In [ ]:
print("\n" + "=" * 70)
print("POINT 5 : Encodage one-hot de 'type'")
print("=" * 70)
 
print(f"Modalités de 'type' avant encodage : {data1['type'].unique().tolist()}")
 
data1_encode = pd.get_dummies(data1, columns=["type"], prefix="type", dtype="int8")
 
nouvelles_colonnes_type = [c for c in data1_encode.columns if c.startswith("type_")]
print(f"Colonnes créées par one-hot encoding : {nouvelles_colonnes_type}")
 
data1 = data1_encode
del data1_encode

## POINT 6 — FEATURES D'AGRÉGATION PAR AGENT (si le volume le permet)

In [ ]:
# Sur un échantillon de ~1.4M lignes (vs 21M), c'est faisable en mémoire.
# ATTENTION méthodologique : ces agrégations DOIVENT être calculées sur
# le train set uniquement puis appliquées (merge) sur val/test, sinon
# fuite de données (data leakage). Ici on montre le calcul sur l'échantillon
# complet à titre exploratoire -- à recalculer correctement après le split
# dans le prochain script (train/val/test).
print("\n" + "=" * 70)
print("POINT 6 : Features d'agrégation par agent (exploratoire)")
print("=" * 70)
 
nb_orig_uniques = data1["nameOrig"].nunique()
nb_dest_uniques = data1["nameDest"].nunique()
print(f"Émetteurs uniques : {nb_orig_uniques:,} sur {len(data1):,} lignes")
print(f"Destinataires uniques : {nb_dest_uniques:,} sur {len(data1):,} lignes")
 
ratio_repetition_orig = len(data1) / nb_orig_uniques
ratio_repetition_dest = len(data1) / nb_dest_uniques
print(f"Ratio moyen lignes/émetteur : {ratio_repetition_orig:.2f}")
print(f"Ratio moyen lignes/destinataire : {ratio_repetition_dest:.2f}")
 
if ratio_repetition_orig < 1.05 and ratio_repetition_dest < 1.05:
    print("\nATTENTION : les identifiants sont quasi tous uniques (ratio ~1).")
    print("Sur des données synthétiques, chaque client n'apparaît généralement")
    print("qu'une seule fois -> les agrégations par agent (nb transactions,")
    print("montant moyen, fréquence) seront peu informatives ou constantes.")
    print("Elles ne sont PAS calculées ici pour éviter d'ajouter du bruit/")
    print("des colonnes inutiles. Si tu confirmes ce constat, documente-le")
    print("explicitement dans le rapport comme justification de l'omission")
    print("(le sujet autorise cette feature 'si le volume le permet').")
else:
    print("\nDes identifiants se répètent suffisamment -> calcul des features")
    print("d'agrégation par agent :")
 
    agg_orig = data1.groupby("nameOrig").agg(
        nb_transactions_orig=("amount", "count"),
        montant_moyen_orig=("amount", "mean")
    ).reset_index()
    data1 = data1.merge(agg_orig, on="nameOrig", how="left")
 
    agg_dest = data1.groupby("nameDest").agg(
        nb_transactions_dest=("amount", "count"),
        montant_moyen_dest=("amount", "mean")
    ).reset_index()
    data1 = data1.merge(agg_dest, on="nameDest", how="left")
 
    print(data1[["nb_transactions_orig", "montant_moyen_orig",
                 "nb_transactions_dest", "montant_moyen_dest"]].describe())

## POINT 7 — SUPPRESSION DES IDENTIFIANTS

In [ ]:
print("\n" + "=" * 70)
print("POINT 7 : Suppression des identifiants nameOrig / nameDest")
print("=" * 70)
 
colonnes_a_supprimer = ["nameOrig", "nameDest"]
if "fichier_source" in data1.columns:
    colonnes_a_supprimer.append("fichier_source")  # colonne technique, pas une feature
 
colonnes_a_supprimer = [c for c in colonnes_a_supprimer if c in data1.columns]
data1 = data1.drop(columns=colonnes_a_supprimer)
print(f"Colonnes supprimées : {colonnes_a_supprimer}")

## RÉCAPITULATIF FINAL

In [ ]:
print("\n" + "=" * 70)
print("RÉCAPITULATIF DU FEATURE ENGINEERING")
print("=" * 70)
print(f"Dimensions finales : {data1.shape[0]:,} lignes, {data1.shape[1]} colonnes")
print(f"\nListe complète des colonnes finales :")
print(data1.columns.tolist())
 
print(f"\nEmpreinte mémoire finale : {data1.memory_usage(deep=True).sum() / 1024**2:.1f} Mo")
 
# Vérification finale : aucune valeur infinie introduite par les divisions
# (amount + 1 protège déjà contre la division par zéro, mais on vérifie)
colonnes_numeriques_finales = data1.select_dtypes(include=[np.number]).columns
nb_infinis = np.isinf(data1[colonnes_numeriques_finales]).sum().sum()
print(f"\nValeurs infinies détectées (doivent être 0) : {nb_infinis}")
if nb_infinis > 0:
    print("ATTENTION : des valeurs infinies existent, à nettoyer avant modélisation.")

## SAUVEGARDE

In [ ]:
data1.to_csv(CHEMIN_SORTIE, index=False)
print(f"\nDataset avec features sauvegardé : {CHEMIN_SORTIE}")
print("\nProchaine étape : split train/val/test stratifié (préserver le ratio")
print("de fraude dans chaque partition), PUIS calcul des agrégations par agent")
print("(point 6) sur le train uniquement si elles sont retenues, pour éviter")
print("toute fuite de données vers val/test.")